In [46]:
from pathlib import Path
import time
import numpy as np

from rag_utils import build_chunks, load_chunks, project_paths, preview, save_chunks
from sentence_transformers import SentenceTransformer

### The model

`all-MiniLM-L6-v2`: 6 transformer layers, 22M parameters, 384 dimensions, ~90 MB.
Small, fast on CPU, and good enough that you should not reach for anything bigger
until you have measured that it is your bottleneck.

[0.0213,-0.0564,0.0089...]

In [47]:
# First run downloads ~90 MB from the Hugging Face hub, then it is cached.
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentence = "Python is a programming langage"
vector = model.encode(sentence)

print("model dimension :", model.get_sentence_embedding_dimension())
print("vector shape    :", vector.shape)
print("vector dtype    :", vector.dtype)
print("length (L2 norm):", float(np.linalg.norm(vector)))
print("\nfirst 8 of 384 numbers:")
print(np.round(vector[:8], 4))
print(f"\nmemory: {vector.nbytes} bytes per chunk "
      f"-> {vector.nbytes * 1_000_000 / 1e9:.1f} GB for a million chunks")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1266.75it/s]


model dimension : 384
vector shape    : (384,)
vector dtype    : float32
length (L2 norm): 1.0

first 8 of 384 numbers:
[-0.0357  0.0203 -0.0289 -0.0007 -0.0699 -0.1152  0.0667  0.0503]

memory: 1536 bytes per chunk -> 1.5 GB for a million chunks


C:\Users\skhal\AppData\Local\Temp\ipykernel_31156\1370385756.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("model dimension :", model.get_sentence_embedding_dimension())


### Experiment 1.1 — does the geometry actually match meaning?

Six sentences. Two are about programming, two about animals, two about the
weather — but they are worded so that **the pairs share almost no vocabulary**.
If embeddings work, the pairs land close together anyway.

In [48]:
sentences = [
    "Python is a programming language",          # 0
    "I write software using python as a programming language",               # 1  (same topic as 0, no shared words)
    "The cat sat on the mat",                    # 2
    "A feline rested on the rug",                # 3  (same meaning as 2, zero shared words)
    "It is raining heavily today",               # 4
    "The weather is wet and stormy",             # 5
]

vectors = model.encode(sentences, normalize_embeddings=True)   # unit length: see section 3
similarity = vectors @ vectors.T                               # all pairs at once

print("     " + "".join(f"{i:>7}" for i in range(len(sentences))))
for i, row in enumerate(similarity):
    print(f"  {i}  " + "".join(f"{value:>7.2f}" for value in row))

print("\n 0 vs 1 (programming pair) :", f"{similarity[0, 1]:.3f}")
print(" 2 vs 3 (animal pair)      :", f"{similarity[2, 3]:.3f}")
print(" 4 vs 5 (weather pair)     :", f"{similarity[4, 5]:.3f}")
print(" 0 vs 3 (unrelated)        :", f"{similarity[0, 3]:.3f}")

           0      1      2      3      4      5
  0     1.00   0.85   0.03   0.06   0.01   0.01
  1     0.85   1.00   0.01  -0.00  -0.02   0.03
  2     0.03   0.01   1.00   0.56   0.04  -0.02
  3     0.06  -0.00   0.56   1.00   0.03  -0.02
  4     0.01  -0.02   0.04   0.03   1.00   0.63
  5     0.01   0.03  -0.02  -0.02   0.63   1.00

 0 vs 1 (programming pair) : 0.847
 2 vs 3 (animal pair)      : 0.564
 4 vs 5 (weather pair)     : 0.631
 0 vs 3 (unrelated)        : 0.063


In [49]:
# The same six sentences judged by keyword overlap, which is what a classic
# search index does. Compare the two numbers for the "cat/feline" pair.
def keyword_overlap(a, b):
    """Jaccard similarity over lowercase word sets."""
    set_a, set_b = set(a.lower().split()), set(b.lower().split())
    return len(set_a & set_b) / len(set_a | set_b)

print(f"{'pair':<46}{'keywords':>10}{'embedding':>11}")
print("-" * 67)
for i, j in [(0, 1), (2, 3), (4, 5), (0, 3)]:
    label = f"{sentences[i][:20]!r} vs {sentences[j][:20]!r}"
    print(f"{label:<46}{keyword_overlap(sentences[i], sentences[j]):>10.2f}"
          f"{similarity[i, j]:>11.2f}")

print("\n'The cat sat on the mat' and 'A feline rested on the rug' share exactly")
print("one word ('on'). Keyword search scores them near zero. The embedding does not.")
print("That gap is the entire reason RAG uses vectors instead of an inverted index.")

pair                                            keywords  embedding
-------------------------------------------------------------------
'Python is a programm' vs 'I write software usi'      0.40       0.85
'The cat sat on the m' vs 'A feline rested on t'      0.22       0.56
'It is raining heavil' vs 'The weather is wet a'      0.10       0.63
'Python is a programm' vs 'A feline rested on t'      0.10       0.06

'The cat sat on the mat' and 'A feline rested on the rug' share exactly
one word ('on'). Keyword search scores them near zero. The embedding does not.
That gap is the entire reason RAG uses vectors instead of an inverted index.


---
## 2. Embed the knowledge base

Now the real thing: every chunk becomes a vector. Two implementation details
that matter more than they look:

- **Batching.** `model.encode` on a list processes chunks in batches on the
  GPU/CPU at once. Calling it 44 times in a Python loop is several times slower
  for identical output.
- **`normalize_embeddings=True`.** Every vector comes back with length 1. Section
  3 explains why that single flag simplifies everything downstream.

In [50]:
paths = project_paths()
chunks = load_chunks(paths["chunks"])

texts = [chunk["text"] for chunk in chunks]

started = time.perf_counter()
embeddings = model.encode(texts, batch_size=32, convert_to_numpy=True,
                          normalize_embeddings=True, show_progress_bar=False)
embeddings = embeddings.astype("float32")      # float32: half the memory of float64,
elapsed = time.perf_counter() - started        # and what FAISS expects anyway

print(f"embedded {len(texts)} chunks in {elapsed:.2f} s "
      f"({len(texts) / elapsed:.0f} chunks/second)")
print("matrix shape :", embeddings.shape, "  <- (n_chunks, n_dimensions)")
print("dtype        :", embeddings.dtype)
print("memory       :", f"{embeddings.nbytes / 1024:.0f} KB")
print("all unit length?", np.allclose(np.linalg.norm(embeddings, axis=1), 1.0))

embedded 44 chunks in 4.23 s (10 chunks/second)
matrix shape : (44, 384)   <- (n_chunks, n_dimensions)
dtype        : float32
memory       : 66 KB
all unit length? True


### Storing it: text + vector + metadata, together

The record we care about is a triple. Keeping the three parts side by side is
the whole idea of a vector database:

```python
{
    "text": "Attention is the operation that lets a model decide ...",
    "embedding": array([0.02, -0.11, ...], dtype=float32),   # for searching
    "metadata": {"source": "ai_course.pdf", "page": 3}       # for citing
}
```

In memory we keep them as **two aligned structures** rather than one list of
dicts — a list of chunks and a `(n, 384)` matrix — because row `i` of the matrix
must correspond to chunk `i`, and numpy can then score all 44 rows in one
operation instead of 44.

That alignment is a real invariant: break it and your citations point at the
wrong document while everything still *looks* like it works.

In [51]:
# A single record, for illustration
record = {
    "text": chunks[0]["text"],
    "embedding": embeddings[0],
    "metadata": chunks[0]["metadata"],
}
print("text     :", preview(record["text"], 120))
print("embedding:", np.round(record["embedding"][:6], 4), "...")
print("metadata :", record["metadata"])

# Persist with numpy + json: two boring formats you can inspect by hand.
np.save(paths["vectors"], embeddings)
save_chunks(chunks, paths["chunks"])
print(f"\nsaved {paths['vectors'].name} ({paths['vectors'].stat().st_size / 1024:.0f} KB)")

# Reload and check the invariant survived the round trip.
reloaded = np.load(paths["vectors"])
assert reloaded.shape == embeddings.shape
assert len(chunks) == len(reloaded)          # rows and chunks must stay aligned
print("round-trip OK:", np.allclose(reloaded, embeddings))

text     : AI Builders Bootcamp - Course Notes These notes cover the core ideas behind modern language models: machine learning, de ...
embedding: [-0.0222 -0.0545  0.0582  0.0264  0.0163  0.0263] ...
metadata : {'source': 'ai_course.pdf', 'path': 'data\\documents\\ai_course.pdf', 'page': 1, 'type': 'pdf', 'chunk_index': 0, 'n_chunks': 6, 'n_chars': 500}

saved vectors.npy (66 KB)
round-trip OK: True


## 3. Similarity search, by hand

### Cosine similarity

cos(angle) = A.B/||A|| x||B||
cos(angle)= A.B
[3,4]
sqrt(3**2 + 4**2)= 25
sqrtt(25)=5

In [52]:
def cosine_similarity(a, b):
    """Cosine similarity between two 1-D vectors, written out in full."""
    a = np.asarray(a, dtype="float32")
    b = np.asarray(b, dtype="float32")
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:                       # a zero vector has no direction
        return 0.0
    return float(np.dot(a, b) / denominator)


a, b = embeddings[0], embeddings[1]
print("our cosine_similarity      :", round(cosine_similarity(a, b), 6))
print("plain dot product (unit)   :", round(float(np.dot(a, b)), 6))

# Cross-check against a library implementation. Never trust your own maths
# until something written by someone else agrees with it.
try:
    from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
    print("sklearn cosine_similarity  :", round(float(sk_cosine([a], [b])[0][0]), 6))
except Exception as error:                 # scikit-learn/scipy unavailable
    print("sklearn unavailable, skipping the cross-check:", type(error).__name__)


our cosine_similarity      : 0.544207
plain dot product (unit)   : 0.544207
sklearn cosine_similarity  : 0.544207


### One query against every chunk

The naive version loops in Python. The vectorised version is one matrix-vector
product: `(44, 384) @ (384,) -> (44,)`. Same numbers, and the gap widens with
every chunk you add.

In [53]:
def cosine_similarity_batch(query_vector, matrix):
    """Cosine similarity of one query against every row of a matrix."""
    query_vector = np.asarray(query_vector, dtype="float32")
    matrix = np.asarray(matrix, dtype="float32")
    denominator = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vector)
    denominator[denominator == 0] = 1e-12          # never divide by zero
    return (matrix @ query_vector) / denominator   # <- the whole search, one line


query_vector = model.encode("What is machine learning?", normalize_embeddings=True)

loop_scores = np.array([cosine_similarity(query_vector, row) for row in embeddings])
batch_scores = cosine_similarity_batch(query_vector, embeddings)
print("loop and vectorised agree:", np.allclose(loop_scores, batch_scores, atol=1e-5))

started = time.perf_counter()
for _ in range(50):
    [cosine_similarity(query_vector, row) for row in embeddings]
loop_ms = (time.perf_counter() - started) * 1000 / 50

started = time.perf_counter()
for _ in range(50):
    cosine_similarity_batch(query_vector, embeddings)
batch_ms = (time.perf_counter() - started) * 1000 / 50

print(f"python loop : {loop_ms:.3f} ms")
print(f"vectorised  : {batch_ms:.3f} ms   ({loop_ms / batch_ms:.0f}x faster on {len(chunks)} chunks)")

loop and vectorised agree: True
python loop : 1.160 ms
vectorised  : 0.084 ms   (14x faster on 44 chunks)


### Top-k

We do not want scores, we want the **best few**. `np.argsort` sorts all *n*
scores; `np.argpartition` finds the top *k* in linear time and only sorts those.
On 44 chunks it makes no difference. On 44 million it is the difference between
a slow query and a fast one.

user query > embedding model > query vector -> compare with document vectors -> similarity scores -> select top-k ->send to the LLM 

In [54]:
def top_k_indices(scores, k):
    """Indices of the k highest scores, best first."""
    k = min(k, len(scores))
    partition = np.argpartition(-scores, k - 1)[:k]   # top k on the left, unordered
    return partition[np.argsort(-scores[partition])]  # then sort just those k


def search(query, chunks, embeddings, top_k=5, min_score=None):
    """The complete retrieval step, in eight lines."""
    query_vector = model.encode(query, normalize_embeddings=True)
    scores = cosine_similarity_batch(query_vector, embeddings)
    hits = []
    for rank, index in enumerate(top_k_indices(scores, top_k), start=1):
        if min_score is not None and scores[index] < min_score:
            continue
        chunk = chunks[index]
        hits.append({"rank": rank, "score": float(scores[index]), "index": int(index),
                     "text": chunk["text"], "metadata": chunk["metadata"]})
    return hits


def show(hits, width=220):
    if not hits:
        print("  (nothing above the score threshold)")
        return
    for hit in hits:
        meta = hit["metadata"]
        page = f", page {meta['page']}" if meta.get("page") else ""
        print(f"  [{hit['rank']}] score={hit['score']:.3f}  ({meta['source']}{page})")
        print(f"      {preview(hit['text'], width)}\n")


print("QUERY: What is machine learning?\n")
show(search("What is machine learning?", chunks, embeddings, top_k=5, min_score=0.55))

QUERY: What is machine learning?



  [1] score=0.605  (ai_course.pdf, page 1)
      ules written by a developer. A traditional program encodes the rules directly; a machine learning system is shown examples and derives the rules on its own by adjusting internal parameters. There are three classic famili ...

  [2] score=0.579  (ai_course.pdf, page 1)
      AI Builders Bootcamp - Course Notes These notes cover the core ideas behind modern language models: machine learning, deep learning, neural networks, transformers, attention, embeddings and retrieval augmented generation ...

  [3] score=0.553  (ai_course.pdf, page 1)
      oss. This procedure is called gradient descent. - Overfitting: the model memorises the training data and fails on new data. - Underfitting: the model is too simple to capture the pattern in the data. - Generalisation: pe ...



---
## 4. FAISS: when 12 lines of numpy stop being enough

Our search scores every vector, every time. That is **O(n · d)** per query:

| chunks | matrix size | one brute-force query |
|---|---|---|
| 44 | 66 KB | microseconds |
| 100,000 | 154 MB | ~10–50 ms |
| 10,000,000 | 15 GB | seconds — and it no longer fits in RAM |

Two separate problems appear as you scale: **speed** and **memory**. FAISS
(Facebook AI Similarity Search) solves both — it is a C++ library of index
structures for exactly this operation.

The key trade-off:

- **`IndexFlatIP`** — exact. Same results as our numpy code, just faster
  (SIMD, multithreaded, cache-aware). Still O(n).
- **`IndexIVFFlat`, `IndexHNSW`, ...** — *approximate*. They avoid looking at
  most vectors, at the cost of occasionally missing a true nearest neighbour.
  Sub-linear, and this is how billion-scale search works.

`IP` = inner product. Our vectors are unit length, so inner product *is* cosine
similarity — the normalisation from section 3 paying off again.

(44,384)
(384,)
(1,384)

In [55]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)   # "flat" = store every vector, compare all
index.add(embeddings)                  # vectors must be float32 and C-contiguous
print("vectors in index:", index.ntotal)

question = "What is machine learning?"
query_vector = model.encode(question, normalize_embeddings=True)
query_matrix = query_vector.reshape(1, -1).astype("float32")   # FAISS takes a batch

scores, indices = index.search(query_matrix, 3)     # -> (1, 3) scores, (1, 3) row ids
print("\nFAISS scores :", np.round(scores[0], 4))
print("FAISS indices:", indices[0])

manual = search(question, chunks, embeddings, top_k=3)
print("numpy scores :", np.round([h["score"] for h in manual], 4))
print("numpy indices:", [h["index"] for h in manual])
print("\nidentical results:", list(indices[0]) == [h["index"] for h in manual])

vectors in index: 44

FAISS scores : [0.6053 0.5788 0.5528]
FAISS indices: [1 0 3]
numpy scores : [0.6053 0.5788 0.5528]
numpy indices: [1, 0, 3]

identical results: True


### Experiment 4.1 — at what scale does it start to matter?

44 chunks is nothing. Let us fake a realistic corpus: 100,000 random 384-d unit
vectors (about a 400-page book collection, or a mid-sized company wiki) and time
the same three approaches. The vectors are random, so the *results* are
meaningless — only the timings matter here.

In [56]:
N = 100_000
rng = np.random.default_rng(0)
big = rng.normal(size=(N, dimension)).astype("float32")
big /= np.linalg.norm(big, axis=1, keepdims=True)          # make them unit length
probe = big[12345].copy()

print(f"{N:,} vectors x {dimension} dims = {big.nbytes / 1e6:.0f} MB\n")

# 1. our numpy brute force
started = time.perf_counter()
for _ in range(10):
    top_k_indices(big @ probe, 5)
numpy_ms = (time.perf_counter() - started) * 100

# 2. FAISS exact
flat = faiss.IndexFlatIP(dimension)
flat.add(big)
started = time.perf_counter()
for _ in range(10):
    flat.search(probe.reshape(1, -1), 5)
flat_ms = (time.perf_counter() - started) * 100
_, exact_ids = flat.search(probe.reshape(1, -1), 5)

# 3. FAISS approximate (IVF): cluster the space, then search only a few clusters
quantizer = faiss.IndexFlatIP(dimension)
ivf = faiss.IndexIVFFlat(quantizer, dimension, 256, faiss.METRIC_INNER_PRODUCT)
ivf.train(big)                      # <- approximate indexes must be trained first
ivf.add(big)
ivf.nprobe = 8                      # search 8 of the 256 clusters (~3% of the data)
started = time.perf_counter()
for _ in range(10):
    ivf.search(probe.reshape(1, -1), 5)
ivf_ms = (time.perf_counter() - started) * 100
_, approx_ids = ivf.search(probe.reshape(1, -1), 5)

recall = len(set(exact_ids[0]) & set(approx_ids[0])) / 5

print(f"{'method':<28}{'ms/query':>10}{'exact?':>9}")
print("-" * 47)
print(f"{'numpy brute force':<28}{numpy_ms:>10.2f}{'yes':>9}")
print(f"{'faiss IndexFlatIP':<28}{flat_ms:>10.2f}{'yes':>9}")
print(f"{'faiss IndexIVFFlat (nprobe=8)':<28}{ivf_ms:>10.2f}{'no':>9}")
print(f"\nIVF recall@5 vs exact: {recall:.0%}  "
      f"(it missed {5 - int(recall * 5)} of the 5 true nearest neighbours)")
print("\nTry nprobe = 1, 8, 32, 256 and watch recall and latency move together.")

100,000 vectors x 384 dims = 154 MB

method                        ms/query   exact?
-----------------------------------------------
numpy brute force                 6.49      yes
faiss IndexFlatIP                19.81      yes
faiss IndexIVFFlat (nprobe=8)      2.49       no

IVF recall@5 vs exact: 60%  (it missed 2 of the 5 true nearest neighbours)

Try nprobe = 1, 8, 32, 256 and watch recall and latency move together.


---
## 5. Retrieval experiments

Retrieval quality is not a vibe — you can look at it. Run the questions below
and read the chunks that come back, asking one question each time:

> **If an LLM saw only these chunks, could it answer?**

That is the only definition of "good retrieval" that matters for RAG.

In [57]:
questions = [
    "What is deep learning?",
    "Explain transformers",
    "What is the attention mechanism?",
    "How do I install FAISS?",
    "Why does my vector search return irrelevant results?",
]

for question in questions:
    print("=" * 78)
    print("QUERY:", question)
    print("=" * 78)
    show(search(question, chunks, embeddings, top_k=2), width=180)

QUERY: What is deep learning?
  [1] score=0.636  (ai_course.pdf, page 1)
      oss. This procedure is called gradient descent. - Overfitting: the model memorises the training data and fails on new data. - Underfitting: the model is too simple to capture the p ...

  [2] score=0.591  (ai_course.pdf, page 1)
      e input and the output. Each layer transforms its input into a slightly more abstract representation, so early layers capture simple local patterns such as edges or character shape ...

QUERY: Explain transformers
  [1] score=0.485  (ai_course.pdf, page 2)
      ross space and excel at images. - Recurrent networks process sequences step by step and carry a hidden state. - Transformers process a whole sequence in parallel using attention. C ...

  [2] score=0.427  (ai_course.pdf, page 2)
      r; it reads the entire sequence at once and lets every token look at every other token. This makes training highly parallel on GPUs and removes the long dependency chains that made ...

QU


### Experiment 5.1 — top_k = 1 versus top_k = 5

In [58]:
question = "Explain transformers"

for k in (1, 5):
    hits = search(question, chunks, embeddings, top_k=k)
    context_chars = sum(len(h["text"]) for h in hits)
    print("=" * 78)
    print(f"top_k = {k}   ->  {context_chars} characters of context "
          f"(~{context_chars // 4} tokens)")
    print("=" * 78)
    for hit in hits:
        meta = hit["metadata"]
        page = f" p{meta['page']}" if meta.get("page") else ""
        print(f"  [{hit['rank']}] {hit['score']:.3f}  {meta['source']}{page}  "
              f"{preview(hit['text'], 110)}")
    print()


top_k = 1   ->  500 characters of context (~125 tokens)
  [1] 0.485  ai_course.pdf p2  ross space and excel at images. - Recurrent networks process sequences step by step and carry a hidden state.  ...

top_k = 5   ->  2071 characters of context (~517 tokens)
  [1] 0.485  ai_course.pdf p2  ross space and excel at images. - Recurrent networks process sequences step by step and carry a hidden state.  ...
  [2] 0.427  ai_course.pdf p2  r; it reads the entire sequence at once and lets every token look at every other token. This makes training hi ...
  [3] 0.232  ai_course.pdf p2  e, positional information must be added explicitly, either as learned positional embeddings or with a scheme s ...
  [4] 0.215  ai_course.pdf p1  e input and the output. Each layer transforms its input into a slightly more abstract representation, so early ...
  [5] 0.203  ai_course.pdf p2  s T5 map one sequence to another and suit translation or summarisation.



### Experiment 5.2 — the question your corpus cannot answer


In [59]:
in_corpus = "What is the attention mechanism?"
out_of_corpus = "How do I configure a Kubernetes ingress controller?"

for question in (in_corpus, out_of_corpus):
    hits = search(question, chunks, embeddings, top_k=5)
    scores = [h["score"] for h in hits]
    print(f"{question}")
    print(f"   scores: {np.round(scores, 3).tolist()}   best={max(scores):.3f}")
    print(f"   best chunk: {preview(hits[0]['text'], 110)}\n")

print("The out-of-corpus question still gets five chunks and a positive score.")
print("The signal is not 'did we get results' - it is 'how high is the top score'.\n")

THRESHOLD = 0.35          # calibrate this on YOUR corpus; it is not universal
for question in (in_corpus, out_of_corpus):
    hits = search(question, chunks, embeddings, top_k=5, min_score=THRESHOLD)
    verdict = f"{len(hits)} chunk(s) pass" if hits else "nothing passes -> answer 'I don't know'"
    print(f"  min_score={THRESHOLD}: {verdict:<45} <- {question[:42]}")

What is the attention mechanism?
   scores: [0.794, 0.496, 0.485, 0.477, 0.425]   best=0.794
   best chunk: Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which ...

How do I configure a Kubernetes ingress controller?
   scores: [0.168, 0.151, 0.138, 0.096, 0.095]   best=0.168
   best chunk: econds. 11. HOUSE DEFAULTS Settings we use unless a project has measured a reason to differ: embedding model s ...

The out-of-corpus question still gets five chunks and a positive score.
The signal is not 'did we get results' - it is 'how high is the top score'.

  min_score=0.35: 5 chunk(s) pass                               <- What is the attention mechanism?
  min_score=0.35: nothing passes -> answer 'I don't know'       <- How do I configure a Kubernetes ingress co


### Experiment 5.3 — chunk size, measured


In [61]:
from rag_utils import load_documents, clean_documents, chunk_documents

documents = clean_documents(load_documents(paths["documents"]))
probe_question = "What is the attention mechanism?"
probe_vector = model.encode(probe_question, normalize_embeddings=True)

print(f"QUESTION: {probe_question}\n")
for size in (200, 500, 1000):
    variant = chunk_documents(documents, chunk_size=size, chunk_overlap=size // 10)
    variant_vectors = model.encode([c["text"] for c in variant],
                                   normalize_embeddings=True).astype("float32")
    scores = cosine_similarity_batch(probe_vector, variant_vectors)
    best = top_k_indices(scores, 1)[0]
    print("=" * 78)
    print(f"chunk_size={size:<5} {len(variant):>3} chunks   best score {scores[best]:.3f}")
    print("=" * 78)
    print(preview(variant[best]["text"], 420))
    print()


QUESTION: What is the attention mechanism?

chunk_size=200   105 chunks   best score 0.827
Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which other tokens matter. Each token is projected into three vectors: a query, a key and a val

chunk_size=500    44 chunks   best score 0.794
Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which other tokens matter. Each token is projected into three vectors: a query, a key and a value. The relevance of token j to token i is the dot product between the query of i and the key of j. Those scores are divided by the square root of the head dimension to keep them numerically stable, passed through a soft ...

chunk_size=1000   23 chunks   best score 0.673
Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which other tokens matter. Each token is projected into three vectors: a query, a 

---
## 6. Packaging it: `rag_store.py`

Everything above now lives in a module next to `rag_utils.py`:

- `Embedder` - the model, always returning unit vectors
- `cosine_similarity` / `cosine_similarity_batch` / `top_k_indices` - the maths, unchanged
- `VectorStore` - chunks + vectors + `search_numpy` / `search_faiss` / `save` / `load`
- `build_or_load_store(paths)` - folder of documents -> searchable store, in one call

Session 1 handed us `chunks`. Session 2 hands on a store. Session 3 (generation)
imports it instead of re-deriving it.

In [ ]:
# Session 2, packaged: Embedder + VectorStore. This is your vector database.
# Everything below runs against rag_store.py, not the definitions above - if this
# cell passes, Session 3 can `from rag_store import ...` and start right here.
import importlib

import rag_store
importlib.reload(rag_store)                      # pick up edits without restarting

from rag_store import Embedder, VectorStore, format_hits

embedder = Embedder()                            # all-MiniLM-L6-v2, cached
store = VectorStore.from_chunks(chunks, embedder, show_progress_bar=False)
store.build_faiss()
store.save(paths["storage"])

print(f"store: {len(store)} chunks, {store.dimension} dims, model {store.model_name}")
print(f"saved to {paths['storage']}")
print()

query_vector = embedder.encode_query("Explain the attention mechanism")
print(format_hits(store.search(query_vector, top_k=3), width=160))

# Prove the two backends agree, and that a saved store reloads identically.
numpy_hits = store.search(query_vector, top_k=5, backend="numpy")
faiss_hits = store.search(query_vector, top_k=5, backend="faiss")
assert [h["index"] for h in numpy_hits] == [h["index"] for h in faiss_hits]
print()
print("numpy and faiss agree on the top 5")

restored = VectorStore.load(paths["storage"])
assert len(restored) == len(store)
assert np.allclose(restored.vectors, store.vectors)
print("saved store reloads identically:", len(restored), "chunks")
print()
print("Session 3 starts from exactly this object.")